Realizado por:
Andrea Bonilla
July Aparicio
Ioshua Castillejos

# Clasificación de imágenes médicas con CNNs
## A - Objetivo

Desarrollar un método basado en redes neuronales convolucionales que permita clasificar con exactitud imágenes extraidas de MRIs en una de 4 categorias. (Glioma, meningioma, pituitary y tejido sano)

## B - Conjunto de datos

Los datos corresponden al **BBC Articles Dataset**, una colección de artículos de
noticias de la BBC etiquetados por categoría (business, entertainment, politics,
sport y tech). El dataset se descarga directamente desde Kaggle usando `kagglehub`
y se guarda en la carpeta `Transformer`.

In [ ]:
import os
import shutil
import kagglehub

# Descarga del dataset desde Kaggle (queda en la cache local de kagglehub)
src_path = kagglehub.dataset_download("jacopoferretti/bbc-articles-dataset")
print("Ruta de descarga:", src_path)

# Copiamos los archivos a la carpeta Transformer (junto a este notebook)
dest_path = os.path.join("bbc-articles-dataset")
if os.path.exists(dest_path):
    shutil.rmtree(dest_path)
shutil.copytree(src_path, dest_path)
print("Dataset guardado en:", os.path.abspath(dest_path))

# Listamos el contenido del dataset
for root, dirs, files in os.walk(dest_path):
    nivel = root.replace(dest_path, "").count(os.sep)
    print("  " * nivel + os.path.basename(root) + "/")
    for f in files:
        print("  " * (nivel + 1) + f)

## C - Actividades por realizar
1. Preparación de las imágenes para el entrenamiento y prueba del modelo.
Construya un pipeline para cargar las imágenes y dividirlas en conjuntos de
entrenamiento y prueba.
2. Desarrollo de la arquitectura de red neuronal convolucional para clasificación
de las imágenes de MRI en una de las 4 categorías. La arquitectura es de libre
elección, se recomienda utilizar bloques de convolución y max pooling o usar
redes existentes y hacerles finetunning para este problema.

### 1. Librerias
Importación de librerias torch para crear los modelos de redes neuronales artificiales.

In [49]:
#Importamos las librerias requeridas
#Importamos pytorch con los modulos para construir redes neuronales

import copy
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split
import torch.optim as optim      # Optimizadores de PyTorch

#Importamos pandas y numpy para trabajar los datos matriciales
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

#Inicializamos las semillas
torch.manual_seed(42)
np.random.seed(42)

#Se define el uso de GPU y cuda para el procesamiento de las redes convolucionales
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cuda


### 2. Preparación y exploración de los datos

#### 2.1 Variables globales

Definimos las variables globales que usaremos durante todo el modelo.
1) Las 4 categorias, en el mismo orden en que ImageFolder lee las carpetas (orden alfabetico).
2) El tamaño de imagen de 150x150 pixeles. Como las imagenes vienen en tamaños muy distintos (ver 2.4) hay que llevarlas todas a uno solo. Al principio se trabajó con 224x224, pero se cambió a 150x150 porque el entrenamiento es mas rapido. La desventaja es que se pierde algo de resolución, sobre todo en las imagenes de 512x512.
3) Para el tamaño del batch se tomó 32, porque un batch mas pequeño hace el entrenamiento mas lento y uno mas grande consume mas memoria.
4) La semilla 42, para que la división de los datos y el entrenamiento se puedan repetir.

In [50]:
#Definición de las variables globales
category = ["glioma","healthy","meningioma","pituitary"]
img_size = (150,150)
batch_size = 32
seed = 42

#### 2.2 Carga del dataset y partición de los datos

Usamos la función ImageFolder de torchvision, que carga las imagenes directamente desde las carpetas y le pone a cada una la etiqueta de la carpeta en la que esta. Aqui se carga con transform=None porque en este paso solo se hace la división. Las transformaciones se asignan despues en la sección 3, una para train (con data augmentation) y otra para validación y test.

1) Cargamos las imagenes en la variable dataset.
2) Dividimos los datos con random_split en 80% train, 10% validación y 10% test (5618, 702 y 703 imagenes). Se usa un generador con la semilla para que la división siempre sea la misma.

In [51]:
#Se realiza la carga de los datos, usando la función Image Folder
dataset = torchvision.datasets.ImageFolder(root='archive', transform=None)

#Se definen los tamaños de los conjuntos train,validaiton y test.
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)
test_size  = total_size - train_size - val_size

#Se realiza la divsión de los datos.
trainset,valset,testset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)